# Visualisierung des Kruskal-Algorithmus & Up-Trees mit Weighted Quick-Union

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/ba_algo_th_public/blob/main/assets/exercises/week_11/kruskal_algorithm.ipynb)

Dieses interaktive Notebook visualisiert den **Algorithmus von Kruskal** zur Bestimmung eines **Minimalen Spannbaums (MST, Minimum Spanning Tree)** auf einem ungerichteten, gewichteten Beispielgraphen mit 8 Knoten.

In jeder Iteration wird:
1. Der **Graph** visualisiert: Die aktuell betrachtete Kante, alle bisher in den MST aufgenommenen Kanten sowie verworfene Kanten werden farblich klar hervorgehoben.
2. Der Wald aus **Up-Trees (Disjoint Sets)** unter Nutzung von **Weighted Quick-Union** (Union by Size/Weight) visualisiert: Die Zeiger der Knoten auf ihre Elternknoten (Parent-Pointers) und die Repräsentanten der Zusammenhangskomponenten werden Schritt für Schritt veranschaulicht.
3. Der interne **Array-Zustand** (`parent` und `weight`/`size`) dokumentiert.


## 1. Theoretische Grundlagen

### Kruskal-Algorithmus
Der Kruskal-Algorithmus ist ein **Greedy-Algorithmus** zur Konstruktion eines minimalen Spannbaums:
1. Sortiere alle Kanten des Graphen aufsteigend nach ihrem Gewicht: $e_1, e_2, \dots, e_m$.
2. Initialisiere den Spannbaum $T = \emptyset$. Jeder Knoten befindet sich zu Beginn in einer eigenen Menge (Zusammenhangskomponente).
3. Für jede Kante $e = (u, v)$ in sortierter Reihenfolge:
   - Prüfe mit `find(u)` und `find(v)`, ob $u$ und $v$ bereits zur selben Zusammenhangskomponente gehören.
   - **Falls nicht (`find(u) != find(v)`):** Kante $e$ schließt keinen Kreis. Füge $e$ zu $T$ hinzu und vereinige die beiden Mengen mit `union(u, v)`.
   - **Falls doch (`find(u) == find(v)`):** Kante $e$ würde einen Kreis bilden und wird **verworfen**.
4. Der Algorithmus stoppt, sobald $T$ genau $|V|-1$ Kanten enthält (oder alle Kanten geprüft wurden).

### Up-Trees und Weighted Quick-Union
Disjunkte Mengen (Disjoint-Set / Union-Find) werden in der Vorlesung als **Up-Trees** modelliert:
- Jeder Knoten besitzt einen Zeiger (Parent-Pointer) auf seinen Elternknoten.
- Die Wurzel eines Baums repräsentiert die gesamte Menge und zeigt auf sich selbst (`parent[root] == root` bzw. `-weight` im klassischen Array).
- `find(i)`: Wandere entlang der Parent-Zeiger bis zur Wurzel.
- **Weighted Quick-Union (Union by Size):** Beim Vereinigen zweier Bäume mit Wurzeln $r_1$ und $r_2$ wird der Baum mit **weniger Knoten (geringerem Gewicht)** unter die Wurzel des Baums mit **mehr Knoten** gehängt. Dadurch wird die Höhe des resultierenden Baums auf maximal $O(\log n)$ beschränkt.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import display, Markdown

# Matplotlib-Darstellung konfigurieren
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10


## 2. Implementierung der Datenstruktur: Weighted Quick-Union

Wir implementieren die Datenstruktur für disjunkte Mengen (`DisjointSetWQU`).
- `parent[u]`: Speichert den Elternknoten von `u`. Zeigt ein Knoten auf sich selbst, ist er die Wurzel seiner Menge.
- `weight[u]`: Speichert für jeden Wurzelknoten die Anzahl der Elemente im Teilbaum.


In [ ]:
class DisjointSetWQU:
    """Disjunkte Mengen (Union-Find) mit Up-Trees und Weighted Quick-Union."""

    def __init__(self, elements):
        self.elements = list(elements)
        self.parent = {x: x for x in self.elements}
        self.weight = {x: 1 for x in self.elements}

    def find(self, i):
        """Findet die Wurzel (Repräsentanten) der Menge von Element i."""
        curr = i
        while curr != self.parent[curr]:
            curr = self.parent[curr]
        return curr

    def union(self, u, v):
        """Vereinigt die Mengen von u und v nach Weighted Quick-Union (Union by Size)."""
        root_u = self.find(u)
        root_v = self.find(v)

        if root_u == root_v:
            return False, root_u, root_v  # Bereits in derselben Menge

        # Der kleinere Baum wird unter die Wurzel des größeren Baums gehängt
        if self.weight[root_u] < self.weight[root_v]:
            self.parent[root_u] = root_v
            self.weight[root_v] += self.weight[root_u]
            winner_root, loser_root = root_v, root_u
        else:
            self.parent[root_v] = root_u
            self.weight[root_u] += self.weight[root_v]
            winner_root, loser_root = root_u, root_v

        return True, winner_root, loser_root

    def get_components(self):
        """Gibt ein Dictionary aller Mengen {wurzel: [elemente]} zurück."""
        comps = {}
        for x in self.elements:
            r = self.find(x)
            comps.setdefault(r, []).append(x)
        return comps


## 3. Definition des Beispielgraphen

Wir definieren einen ungerichteten, zusammenhängenden Beispielgraphen mit **8 Knoten** ($A, B, C, D, E, F, G, H$) und **10 gewichteten Kanten** (angelehnt an das Übungsblatt zu minimalen Spannbäumen).
Feste Knotenkoordinaten stellen eine übersichtliche und reproduzierbare Darstellung sicher.


In [ ]:
# Definition der Knoten und Kanten mit Gewichten
nodes = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

# Ungerichtete Kanten: (u, v, gewicht)
edges = [
    ('C', 'E', 1),
    ('F', 'G', 1),
    ('B', 'D', 2),
    ('D', 'H', 2),
    ('A', 'C', 3),
    ('C', 'D', 3),
    ('A', 'G', 3),
    ('D', 'F', 4),
    ('B', 'C', 5),
    ('E', 'F', 5)
]

# Feste 2D-Koordinaten für eine übersichtliche Darstellung des Graphen
graph_layout = {
    'A': (1.0, 0.5),
    'B': (0.0, 2.5),
    'C': (1.2, 1.8),
    'D': (2.8, 2.5),
    'E': (2.4, 1.0),
    'F': (4.0, 2.0),
    'G': (3.6, 0.5),
    'H': (2.8, 3.8)
}

G = nx.Graph()
G.add_nodes_from(nodes)
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

print(f"Graph initialisiert: {G.number_of_nodes()} Knoten, {G.number_of_edges()} Kanten.")


## 4. Visualisierungsfunktionen für Graph und Up-Trees

Wir erstellen zwei spezialisierte Visualisierungsroutinen:
1. `draw_graph_state(...)`: Zeichnet den Graphen.
   - Bisher akzeptierte MST-Kanten: **Grün, dick**
   - Aktuell untersuchte Kante: **Orange/Gold, dick**
   - Verworfene Kanten (die Zyklen gebildet hätten): **Rot gestrichelt**
   - Noch ungeprüfte Kanten: **Hellgrau**
2. `draw_up_trees(...)`: Zeichnet die **Up-Trees** der disjunkten Mengen nebeneinander.
   - Pfeile zeigen vom Kind zum Elternknoten (Parent-Pointer: $	o$).
   - Wurzelknoten (Repräsentanten) werden farblich hervorgehoben und tragen ihre Baumgröße (`Größe/Weight`).


In [ ]:
def compute_up_tree_layout(elements, parent):
    """Berechnet übersichtliche Koordinaten für die Up-Tree-Wälder."""
    roots = sorted([x for x in elements if parent[x] == x])
    children = {x: [] for x in elements}
    for x in elements:
        if parent[x] != x:
            children[parent[x]].append(x)
    for x in children:
        children[x].sort()

    pos = {}
    current_x = 0.0

    for r in roots:
        leaf_count = 0

        def layout_node(u, depth=0):
            nonlocal leaf_count
            if not children[u]:
                leaf_count += 1
                pos[u] = (current_x + leaf_count * 1.6, -depth * 1.4)
            else:
                child_xs = []
                for c in children[u]:
                    layout_node(c, depth + 1)
                    child_xs.append(pos[c][0])
                pos[u] = (sum(child_xs) / len(child_xs), -depth * 1.4)

        layout_node(r, depth=0)

        # Maximales X dieses Baumes bestimmen
        def get_all(u):
            res = [u]
            for c in children[u]:
                res.extend(get_all(c))
            return res

        max_tree_x = max(pos[x][0] for x in get_all(r))
        current_x = max_tree_x + 1.2

    return pos


def draw_iteration(iteration_num, current_edge, decision_info, mst_edges, rejected_edges, remaining_edges, ds):
    """Visualisiert nebeneinander den Graphen (links) und die Up-Trees (rechts)."""
    fig, (ax_graph, ax_trees) = plt.subplots(1, 2, figsize=(16, 6))

    # --- 1. Graph visualisieren (links) ---
    ax_graph.set_title(f"Graph (Iteration {iteration_num})", fontsize=13, fontweight='bold')

    # Ungeprüfte Kanten
    nx.draw_networkx_edges(G, graph_layout, edgelist=remaining_edges,
                           edge_color='#c0c0c0', width=1.5, ax=ax_graph)

    # Verworfene Kanten (Zyklen)
    if rejected_edges:
        nx.draw_networkx_edges(G, graph_layout, edgelist=rejected_edges,
                               edge_color='#e74c3c', width=2.0, style='dotted', ax=ax_graph)

    # MST-Kanten
    if mst_edges:
        nx.draw_networkx_edges(G, graph_layout, edgelist=mst_edges,
                               edge_color='#27ae60', width=4.0, ax=ax_graph)

    # Aktuelle Kante
    if current_edge:
        u, v = current_edge[0], current_edge[1]
        nx.draw_networkx_edges(G, graph_layout, edgelist=[(u, v)],
                               edge_color='#f39c12', width=5.0, ax=ax_graph)

    # Knoten
    nx.draw_networkx_nodes(G, graph_layout, node_color='#3498db', node_size=750, ax=ax_graph)
    nx.draw_networkx_labels(G, graph_layout, font_size=11, font_color='white',
                            font_weight='bold', ax=ax_graph)

    # Kantengewichte
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, graph_layout, edge_labels=edge_labels,
                                 font_size=10, font_weight='bold', ax=ax_graph)

    ax_graph.axis('off')

    # --- 2. Up-Trees visualisieren (rechts) ---
    ax_trees.set_title("Up-Trees (Weighted Quick-Union)", fontsize=13, fontweight='bold')

    tree_pos = compute_up_tree_layout(ds.elements, ds.parent)
    T = nx.DiGraph()
    T.add_nodes_from(ds.elements)

    tree_edges = [(u, ds.parent[u]) for u in ds.elements if ds.parent[u] != u]
    T.add_edges_from(tree_edges)

    # Unterscheidung Wurzeln vs. Kindknoten
    root_nodes = [x for x in ds.elements if ds.parent[x] == x]
    child_nodes = [x for x in ds.elements if ds.parent[x] != x]

    # Wurzeln (Grün) und normale Knoten (Blau)
    nx.draw_networkx_nodes(T, tree_pos, nodelist=root_nodes,
                           node_color='#2ecc71', node_size=800, ax=ax_trees)
    nx.draw_networkx_nodes(T, tree_pos, nodelist=child_nodes,
                           node_color='#5dade2', node_size=700, ax=ax_trees)

    # Beschriftung der Knoten
    labels = {x: f"{x}\n(w={ds.weight[x]})" if x in root_nodes else x for x in ds.elements}
    nx.draw_networkx_labels(T, tree_pos, labels=labels, font_size=9,
                            font_color='black', font_weight='bold', ax=ax_trees)

    # Parent-Pointer-Pfeile (Kind -> Elternknoten)
    if tree_edges:
        nx.draw_networkx_edges(T, tree_pos, edgelist=tree_edges,
                               arrows=True, arrowstyle='-|>', arrowsize=18,
                               edge_color='#34495e', width=2.0, ax=ax_trees,
                               connectionstyle='arc3,rad=0.0')

    ax_trees.margins(0.2)
    ax_trees.axis('off')

    plt.tight_layout()
    plt.show()


## 5. Iteration für Iteration: Kruskals Algorithmus

Nun führen wir den Kruskal-Algorithmus schrittweise aus:
- Zuerst werden alle Kanten aufsteigend nach Gewicht sortiert.
- In jeder Iteration wird die Kante mit dem kleinsten verbleibenden Gewicht geprüft.
- `find(u)` und `find(v)` ermitteln die Repräsentanten der jeweiligen Mengen.
- Bei unterschiedlichen Repräsentanten wird die Kante zum MST hinzugefügt und `union(u, v)` nach Weighted Quick-Union ausgeführt.
- Bei gleichen Repräsentanten wird die Kante verworfen, da sie einen Kreis schließen würde.


In [ ]:
# 1. Kanten aufsteigend nach Gewicht sortieren
sorted_edges = sorted(edges, key=lambda e: (e[2], e[0], e[1]))

print("Sortierte Kantenreihenfolge:")
for i, (u, v, w) in enumerate(sorted_edges, 1):
    print(f"  {i}. ({u}, {v}) mit Gewicht {w}")


### Initialer Zustand vor Beginn der ersten Iteration

Vor der Prüfung der ersten Kante:
- Der minimale Spannbaum ist leer ($T = \emptyset$).
- Jeder Knoten bildet seinen eigenen Up-Tree mit Gewicht 1.


In [ ]:
ds = DisjointSetWQU(nodes)
mst_edges = []
rejected_edges = []
remaining = [(u, v) for (u, v, _) in sorted_edges]

draw_iteration(
    iteration_num=0,
    current_edge=None,
    decision_info="Initialer Zustand: Jeder Knoten in eigener Menge.",
    mst_edges=mst_edges,
    rejected_edges=rejected_edges,
    remaining_edges=remaining,
    ds=ds
)

# Tabellarische Übersicht des Parent- und Weight-Arrays
df_init = pd.DataFrame({
    'Knoten': nodes,
    'Parent': [ds.parent[x] for x in nodes],
    'Gewicht (Wurzel)': [ds.weight[x] if ds.parent[x] == x else '-' for x in nodes]
})
display(df_init.T)


### Ausführung aller Iterationen


In [ ]:
for step, (u, v, w) in enumerate(sorted_edges, 1):
    root_u = ds.find(u)
    root_v = ds.find(v)

    # Restliche noch unbetrachtete Kanten
    remaining = [(x, y) for (x, y, _) in sorted_edges[step:]]

    status_lines = [
        f"### Iteration {step}: Kante ({u}, {v}) mit Gewicht {w}",
        f"- `find('{u}')` = **{root_u}** (Größe des Baums: {ds.weight[root_u]})",
        f"- `find('{v}')` = **{root_v}** (Größe des Baums: {ds.weight[root_v]})"
    ]

    if root_u != root_v:
        # Kein Zyklus -> Kante aufnehmen und union ausführen
        success, winner, loser = ds.union(u, v)
        mst_edges.append((u, v))
        status_lines.append(
            f"- **Entscheidung:** **AKZEPTIERT** (kein Zyklus). Kante ({u}, {v}) wird zum MST hinzugefügt."
        )
        status_lines.append(
            f"- **Weighted Quick-Union:** Wurzel `{loser}` (Größe {ds.weight[winner] - ds.weight[loser]}) wird Kind von Wurzel `{winner}` (neue Größe: {ds.weight[winner]})."
        )
    else:
        # Zyklus -> Kante verwerfen
        rejected_edges.append((u, v))
        status_lines.append(
            f"- **Entscheidung:** **VERWORFEN** (Zyklus erkannt!). Beide Knoten besitzen bereits die gleiche Wurzel `{root_u}`."
        )

    display(Markdown("\n".join(status_lines)))

    # Grafische Darstellung
    draw_iteration(
        iteration_num=step,
        current_edge=(u, v),
        decision_info=f"Iteration {step}: ({u}, {v}, {w})",
        mst_edges=mst_edges,
        rejected_edges=rejected_edges,
        remaining_edges=remaining,
        ds=ds
    )

    # Aktueller Array-Zustand
    df_state = pd.DataFrame({
        'Knoten': nodes,
        'Parent': [ds.parent[x] for x in nodes],
        'Gewicht (Wurzel)': [ds.weight[x] if ds.parent[x] == x else '-' for x in nodes]
    })
    display(df_state.T)
    print("-" * 80)


## 6. Endergebnis und Zusammenfassung

Nach Prüfung aller Kanten (bzw. sobald $|V|-1 = 7$ Kanten im MST sind):
- Alle 8 Knoten des Graphen sind in einer einzigen Zusammenhangskomponente vereinigt.
- Die Up-Tree-Wurzel ist der gemeinsame Repräsentant aller Knoten.
- Der minimale Spannbaum ist vollständig konstruiert.


In [ ]:
total_weight = sum(G[u][v]['weight'] for u, v in mst_edges)

print("=" * 60)
print("ERGEBNIS DES KRUSKAL-ALGORITHMUS:")
print(f"Anzahl Knoten im Graph: {len(nodes)}")
print(f"Kanten im MST ({len(mst_edges)} Kanten):")
for u, v in mst_edges:
    print(f"  • ({u}, {v}) mit Gewicht {G[u][v]['weight']}")
print(f"Gesamtgewicht des Minimalen Spannbaums: {total_weight}")
print("=" * 60)
